# DRAM Bender Python Tutorial

DRAM Bender provides both Python and C++ APIs. This notebook uses the Python API with DDR4 on an Alveo U200 board to explain the following aspects of DRAM Bender and its API:
1. DRAM Bender hardware and programming model
2. How to write custom DRAM Bender test programs
3. How to debug DRAM Bender test programs

This notebook requires an Alveo U200 programmed with the maintained DRAM Bender bitstream and a DDR4 module under test installed on the board.


## 0. DRAM Bender Hardware and Programming Model

From a programmer's perspective, the DRAM Bender hardware contains:

- A RISC-like, in-order core with fixed latency instructions
- A register file
- A on-chip scratchpad memory

A DRAM Bender test program consists of two kinds of instructions:

1. Fabric instructions that the DRAM Bender core itself execute. Fabric instrucions can manipulate the register file and scratchpad memory.
2. DRAM commands that DRAM Bender sends to the DRAM module under test.

Examples of DRAM Bender fabric instructions:

1. Arithematic instructions: `ADD`, `LI`
2. Scratchpad memory load/store: `LD`/`ST`
3. Control flow instructions: `BL`

A fabric instruction occupies one 64-bit instruction word. Non-control flow fabric instructions take 1 fabric cycle. Control flow instructions take 6 fabric cycles. DRAM Bender also provides a `SLEEP` instruction that spins for a speficied number of fabric cycles.

A DRAM command occupies a 16-bit slot in an instruction word (i.e., 16-bit per DRAM command). Each DRAM command takes 1/4 fabric cycle. Note that DRAM Bender operates at fabric cycles, DRAM commands must be `NOP` padded to groups of 4.


## 1. Prerequisite

First, make sure to create the development environment before opening this notebook:

    bash setup_venv.sh

Then select the repository virtual environment as the Jupyter kernel.

The first step to use DRAM Bender is to identify 1) the FPGA board, and 2) the DRAM Bender instance in the OS. The DRAM Bender API uses `PCI_BDF` (e.g., `0000:01:00.0`) to identify an FPGA board, and `XDMA_CHANNEL` to identify DRAM Bender instances on that board. A board may contain one or more DRAM Bender instances (e.g., U200 has four independent DDR4 channels, corresponding to up to 4 independent DRAM Bender instances).

The maintained U200 design exposes one XDMA PCIe function per board, so its BDF serves as the board identifier. On a host with several U200s, each board has a different BDF. Moving a board to another PCIe slot can change its BDF. Find the complete BDF of the programmed U200 before starting Jupyter:

    lspci -D -d 10ee:

The configuration cell below contains example values for <code>PCI_BDF</code> and <code>XDMA_CHANNEL</code>. Replace the BDF with the value reported on the local system. A bitstream can expose more than one XDMA channel; select only a channel backed by an installed DRAM module.



In [1]:
import sys

import numpy as np

import drambender
from drambender.api import (
    DDR4Target,
    FinalProgram,
    HostInterface,
    ProgramBuilder,
    board_configs,
    open_board,
    program_template,
)
from drambender.api.program.instructions import *

assert sys.version_info >= (3, 10)
print(f"Python: {sys.executable}")
print(f"drambender: {drambender.__file__}")

# These example values may need to be changed to match your system.
# Find the board BDF with `lspci -D -d 10ee:` and select an XDMA
# instance connected to the installed DDR4 module under test.
PCI_BDF = "0000:01:00.0"
XDMA_CHANNEL = 0

print(f"Endpoint: {PCI_BDF}, XDMA channel {XDMA_CHANNEL}")


Python: /home/hluo/DRAM-Bender-board-config/.venv/bin/python
drambender: /home/hluo/DRAM-Bender-board-config/python/drambender/__init__.py
Endpoint: 0000:01:00.0, XDMA channel 0


### Target

The DRAM Bender API defines `target`: a collection of immutable properties of both the DRAM Bender hardware and the DRAM under test. They are immutable for the duration of the DRAM Bender program, and should only be changed if you 1) modify DRAM Bender hardware and programs the FPGA with a new bitstream, or 2) you swapped another DRAM module.

For this tutorial, <code>DDR4Target</code> contains:
1. The latency per fabric instruction (and thus, DRAM commands) of the current DRAM Bender hardware.
2. The memory geometry of the DRAM under test (e.g., number rows, number of cachelines per row).

In [2]:
CACHELINES_PER_ROW = 128
WORDS_PER_CACHELINE = 16
COLUMN_STRIDE = 8

TARGET = DDR4Target(
    cachelines_per_row=CACHELINES_PER_ROW,
    column_stride=COLUMN_STRIDE,
    words_per_cacheline=WORDS_PER_CACHELINE,
    rank=0,
)

ROW_WORDS = TARGET.columns_per_row * TARGET.words_per_cacheline
ROW_BYTES = ROW_WORDS * np.dtype(np.uint32).itemsize

assert TARGET.board_config is board_configs.U200
print(f"DDR4 row readback: {ROW_BYTES} bytes")
print(TARGET.board_config.summary())


DDR4 row readback: 8192 bytes
Board:                      U200
Memory type:                DDR4
Instruction capacity:       32768
DRAM command slot:          1.500000 ns
DRAM slots/fabric cycle:    4
Fabric cycle:               6.000000 ns
Readback buffer capacity:   1024 entries


## 2. Your First DRAM Bender Programs

We will first start by looking at a simple DRAM Bender program that writes to and then immediately read back from a DRAM row. The DRAM Bender API maintains a collection of builtin program templates for common functionalities (defined in [python/drambender/builtin_programs/](../python/drambender/builtin_programs/)).

The following code block constructs a write program and read program. It first "binds" the current target to the builtin program templates, and then instaniate a write program and read program.

Building a program does not touch hardware. It returns a FinalProgram that can be held in memory, inspected, reused, and submitted later.


In [3]:
BANK = 0
ROW = 0
PATTERN = 0xDEADBEEF

# Set the builtin program templates for the current target
programs = drambender.builtin_programs.configure(target=TARGET)

pattern_words = (PATTERN,) * TARGET.words_per_cacheline

# Instaniate the write and read program
write_program = programs.write_row(BANK, ROW, pattern_words)
read_program = programs.read_row(BANK, ROW)

print(
    "Instructions:",
    write_program.instruction_count,
    "write,",
    read_program.instruction_count,
    "read",
)


Instructions: 302 write, 271 read


## 3. Inspect, Dry-Run, And Trace A Program

Now, let us inspect the programs constructed above. The DRAM Bender API provides three complementary tools:

1. Pretty-print the program.
2. Execute the program in a software virtual machine and report its execution behavior.
3. Trace every DRAM command with a timestamp and summarize selected DRAM timing intervals.


### 3.1 Pretty-Print The Program

Converting a <code>FinalProgram</code> to a string produces a decoded instruction listing. Scalar instructions appear by themselves, while a packed DRAM instruction shows all four mini-operation slots. The following cell prints the first 30 instructions of the row-read program.


In [5]:
listing_lines = str(read_program).splitlines()
print("\n".join(listing_lines[:30]))
if len(listing_lines) > 30:
    print(f"... {len(listing_lines) - 30} more lines")


0000 | LI BAR, 0
0001 | LI RAR, 0
0002 | LI CASR, 8
0003 | PRE BAR | NOP | NOP | NOP
0004 | LI CAR, 0
0005 | AUTOGEN: INFO read_count=128
0006 | NOP | NOP | NOP | NOP
0007 | NOP | NOP | NOP | NOP
0008 | ACT BAR, RAR | NOP | NOP | NOP
0009 | NOP | NOP | NOP | NOP
0010 | NOP | NOP | NOP | NOP
0011 | RD BAR, CAR++ | NOP | NOP | NOP
0012 | NOP | NOP | NOP | NOP
0013 | RD BAR, CAR++ | NOP | NOP | NOP
0014 | NOP | NOP | NOP | NOP
0015 | RD BAR, CAR++ | NOP | NOP | NOP
0016 | NOP | NOP | NOP | NOP
0017 | RD BAR, CAR++ | NOP | NOP | NOP
0018 | NOP | NOP | NOP | NOP
0019 | RD BAR, CAR++ | NOP | NOP | NOP
0020 | NOP | NOP | NOP | NOP
0021 | RD BAR, CAR++ | NOP | NOP | NOP
0022 | NOP | NOP | NOP | NOP
0023 | RD BAR, CAR++ | NOP | NOP | NOP
0024 | NOP | NOP | NOP | NOP
0025 | RD BAR, CAR++ | NOP | NOP | NOP
0026 | NOP | NOP | NOP | NOP
0027 | RD BAR, CAR++ | NOP | NOP | NOP
0028 | NOP | NOP | NOP | NOP
0029 | RD BAR, CAR++ | NOP | NOP | NOP
... 241 more lines


### 3.2 Dry-Run The Program

<code>dry_run()</code> executes the program in the software virtual machine without submitting it to the FPGA. It reports the number of executed instructions and cycles, branch behavior, DRAM command counts, and final register values. <code>max_instructions</code> bounds the run so that a program with an unintended infinite loop cannot run forever. The virtual machine checks the instruction stream, but it does not model DRAM data or analog behavior.


In [6]:
result = read_program.dry_run(max_instructions=100_000)
print("Dry-run result")
print(result)
print("DRAM command counts:", result.dram_cmd_counts)


Dry-run result
Execution result:
  Total cycles         : 275
  Total time           : 0.002 ms (0.000002 s)
  Instructions executed: 271
  Branches taken       : 0
  DRAM commands:
    RD      : 128
    PRE     : 2
    ACT     : 1
    NOP     : 921
  Nonzero registers:
    R0 (CASR)        = 0x00000008 (8)
    R3 (CAR)         = 0x00000400 (1024)

DRAM command counts: {'WR': 0, 'RD': 128, 'PRE': 2, 'ACT': 1, 'SEL_CH': 0, 'REF': 0, 'NOP': 921}


### 3.3 Trace DRAM Commands

<code>trace_dram_commands()</code> returns the ordered DRAM commands issued by the program. Each event records its time from the beginning of the program and the elapsed time since the preceding DRAM command. The trace can therefore be used to inspect command placement directly.

<code>trace.summarize_timings()</code> reports the observed tRCD, tRAS, and tRP intervals for each bank. It does not compare them with a device specification and does not cover every DRAM timing constraint, so compare the reported values with the requirements of the module under test.


In [7]:
trace = read_program.trace_dram_commands(max_instructions=100_000)
assert not trace.truncated
print("First DRAM command events")
for event in trace.events[:12]:
    print(
        f"t={event.time_ns:7.1f} ns, "
        f"delta={event.delta_ns:7.1f} ns, "
        f"command={event.command}, bank={event.bank}"
    )

print("\nTiming summary")
print(trace.summarize_timings())


First DRAM command events
t=   18.0 ns, delta=    0.0 ns, command=PRE, bank=0
t=   48.0 ns, delta=   30.0 ns, command=ACT, bank=0
t=   66.0 ns, delta=   18.0 ns, command=RD, bank=0
t=   78.0 ns, delta=   12.0 ns, command=RD, bank=0
t=   90.0 ns, delta=   12.0 ns, command=RD, bank=0
t=  102.0 ns, delta=   12.0 ns, command=RD, bank=0
t=  114.0 ns, delta=   12.0 ns, command=RD, bank=0
t=  126.0 ns, delta=   12.0 ns, command=RD, bank=0
t=  138.0 ns, delta=   12.0 ns, command=RD, bank=0
t=  150.0 ns, delta=   12.0 ns, command=RD, bank=0
t=  162.0 ns, delta=   12.0 ns, command=RD, bank=0
t=  174.0 ns, delta=   12.0 ns, command=RD, bank=0

Timing summary
Timing summary (ns):
  tRCD  min=   18.0  max=   18.0  n=1
  tRAS  min= 1578.0  max= 1578.0  n=1
  tRP   min=   30.0  max=   30.0  n=1


## 4. Execute A DDR4 Read/Write Program

Now we will execute the program on the FPGA with the DRAM under test. The execution workflow is:

1. Open one endpoint with a target, complete PCI BDF, and XDMA channel.
2. Establish a clean session with <code>full_reset()</code>.
3. Submit one program or a list of programs with <code>execute()</code>.
4. Copy the expected number of ordered readback bytes into a writable, C-contiguous NumPy buffer with <code>receive_into()</code>.
5. Call <code>synchronize()</code> to wait for the active readback session and surface asynchronous errors.

The context manager closes the handle and releases endpoint ownership. It does not itself reset the FPGA. Opening the board prints its BoardConfig and reminds you that the programmed bitstream must match. The message reports API assumptions; it does not identify the bitstream automatically.


In [8]:
readback = np.empty(ROW_WORDS, dtype=np.uint32)

with open_board(
    TARGET,
    pci_bdf=PCI_BDF,
    xdma_channel=XDMA_CHANNEL,
    host_interface=HostInterface.XDMA,
) as board:
    board.full_reset()
    board.execute([write_program, read_program])
    board.receive_into(readback, timeout=None)
    board.synchronize()

expected = np.full_like(readback, PATTERN)
mismatches = int(np.count_nonzero(readback != expected))
assert mismatches == 0, f"{mismatches} mismatched words"
print(f"PASS: {readback.size} words matched 0x{PATTERN:08x}")


[DRAM Bender] Board opened
PCIe endpoint:              0000:01:00.0
XDMA channel:               0
Host interface:             XDMA
Board:                      U200
Memory type:                DDR4
Instruction capacity:       32768
DRAM command slot:          1.500000 ns
DRAM slots/fabric cycle:    4
Fabric cycle:               6.000000 ns
Readback buffer capacity:   1024 entries
The API assumes that the programmed bitstream matches this board configuration.


PASS: 2048 words matched 0xdeadbeef


### Reset And Interruption Behavior

Use <code>reset_fpga()</code> for a normal synchronized logic reset. Use <code>full_reset()</code> to cancel active readback, reset FPGA logic, drain stale host data, and clear queued software data.

A <code>receive_into(..., timeout=None)</code> call waits without a deadline, including across long retention intervals. A finite receive timeout, an asynchronous readback error, or Ctrl+C during a main-thread Python receive or synchronization wait triggers a full reset before the exception is raised. If a process is killed or intentionally abandons in-flight work, the next process should begin with <code>full_reset()</code>.


## 5. RowHammer With Built-In Programs

Now let us up our game from simple write-read to single-sided RowHammer.

DRAM Bender provides the `Row` class that encapsulates a physical DRAM row. You can specify its physical row index, its row mapping (we use the no-op `Linear` mapping as a example here), and data pattern.

First, let us build the victim and aggressor row objects, the hammer program, and then trace it.

In [9]:
from drambender.rows import Row, available_mappings

START_ROW = 81
# row_mapping specifies the DRAM bank's internal row mapping.
# This tutorial uses the built-in Linear mapping as an example.
victim = Row(
    physical_id=START_ROW,
    row_mapping="Linear",
    data_pattern=0x00000000,
)
aggressor = Row(
    physical_id=START_ROW + 1,
    row_mapping="Linear",
    data_pattern=0xFFFFFFFF,
)

hammer_preview = programs.single_sided_rowhammer(
    BANK,
    aggressor.logical_id,
    hammer_count=8,
)
hammer_preview_result = hammer_preview.dry_run(max_instructions=10_000)

print("Available row mappings:", available_mappings())
print("Selected row mapping:", victim.row_mapping.name)
print(
    f"victim physical={victim.physical_id}, logical={victim.logical_id}; "
    f"aggressor physical={aggressor.physical_id}, logical={aggressor.logical_id}"
)
print(hammer_preview_result)


Available row mappings: ('linear', 'mi1', 'sa0')
Selected row mapping: Linear
victim physical=81, logical=81; aggressor physical=82, logical=82
Execution result:
  Total cycles         : 80
  Total time           : 0.000 ms (0.000000 s)
  Instructions executed: 39
  Branches taken       : 7
  DRAM commands:
    PRE     : 9
    ACT     : 8
    NOP     : 51
  Nonzero registers:
    R5 (RAR)         = 0x00000052 (82)
    R7               = 0x00000008 (8)
    R8               = 0x00000008 (8)



Now, let us make it into an end-to-end single-sided RowHammer tester that sweeps a range of victim rows, and for each victim row, first initializes the aggressor and victim, then perform single-sided RowHammer, and finally read the victim data back to check for bitflips.

In [10]:
HAMMER_COUNT = 500_000
NUM_VICTIMS = 5
total_bitflips = 0

with open_board(
    TARGET,
    pci_bdf=PCI_BDF,
    xdma_channel=XDMA_CHANNEL,
    host_interface=HostInterface.XDMA,
) as board:
    board.full_reset()

    for offset in range(NUM_VICTIMS):
        victim_physical = START_ROW + offset
        aggressor_physical = victim_physical + 1

        victim = Row(
            physical_id=victim_physical,
            row_mapping="Linear",
            data_pattern=0x00000000,
        )
        aggressor = Row(
            physical_id=aggressor_physical,
            row_mapping="Linear",
            data_pattern=0xFFFFFFFF,
        )

        board.execute([
            programs.write_row(
                BANK, victim.logical_id, victim.write_pattern
            ),
            programs.write_row(
                BANK, aggressor.logical_id, aggressor.write_pattern
            ),
            programs.single_sided_rowhammer(
                BANK, aggressor.logical_id, HAMMER_COUNT
            ),
            programs.read_row(BANK, victim.logical_id),
        ])

        readback = np.empty(ROW_WORDS, dtype=np.uint32)
        board.receive_into(readback, timeout=None)
        board.synchronize()

        row_pattern = np.asarray(victim.write_pattern, dtype=np.uint32)
        expected = np.tile(row_pattern, ROW_WORDS // row_pattern.size)
        mask = readback ^ expected
        bitflips = int(
            np.unpackbits(mask.view(np.uint8), bitorder="little").sum()
        )
        total_bitflips += bitflips
        print(
            f"victim {victim_physical:5d}, "
            f"aggressor {aggressor_physical:5d}: "
            f"{bitflips} bit flips"
        )

print(
    f"Total: {total_bitflips} bit flips across "
    f"{NUM_VICTIMS} victim rows"
)


[DRAM Bender] Board opened
PCIe endpoint:              0000:01:00.0
XDMA channel:               0
Host interface:             XDMA
Board:                      U200
Memory type:                DDR4
Instruction capacity:       32768
DRAM command slot:          1.500000 ns
DRAM slots/fabric cycle:    4
Fabric cycle:               6.000000 ns
Readback buffer capacity:   1024 entries
The API assumes that the programmed bitstream matches this board configuration.


victim    81, aggressor    82: 16 bit flips
victim    82, aggressor    83: 29 bit flips
victim    83, aggressor    84: 21 bit flips
victim    84, aggressor    85: 32 bit flips
victim    85, aggressor    86: 29 bit flips
Total: 127 bit flips across 5 victim rows


Readback is an ordered byte stream. The caller chooses how many bytes to copy into each buffer, so one receive call can consume part of a program's output or data from several queued programs. Buffer sizes must be multiples of four bytes.

Derive readback sizes from the target geometry instead of hard-coding one row size. The U200 DDR4 target above yields 8192 bytes per row read.


## 6. Write A Custom DRAM Bender Program With ProgramBuilder

Now, let us write our own DRAM Bender programs. The DRAM Bender API provides a `ProgramBuilder` that simplifies the authoring of programs. A typical DRAM Bender program workflow is as follows:

1. Create it with the U200 <code>DDR4Target</code>.
2. Load scalar registers with operations such as <code>LI</code> and <code>ADDI</code>.
3. Copy scalar write values into the 16-lane wide write-data register with <code>LDWD</code>.
4. Issue DRAM commands with exact <code>DRAM</code> packing or a timed <code>DRAMSEQ</code>.
5. Add sleeps, labels, and branches.
6. Call <code>conclude()</code> to resolve control flow, append termination, and insert read-count metadata for the FPGA readback engine.

DRAM Bender has 16 scalar registers. Registers 0 through 6 have conventional names: CASR, BASR, RASR, CAR, BAR, RAR, and PATTERN_REG. PATTERN_REG is a scalar staging register; LDWD copies it into the separate wide write-data lanes. Nine additional registers can be allocated by name. ProgramBuilder does not spill registers.


Top-level slot-hiding calls such as <code>p.PRE()</code> and <code>p.ACT()</code> are intentionally forbidden. Use exact packing:

    p.DRAM(PRE("BAR"), NOP(), NOP(), NOP())

For a multi-command timed sequence, delays are mini-slots:

    p.DRAMSEQ(
        PRE("BAR", delay=12),
        ACT("BAR", "RAR", delay=11),
        ALIGN(),
    )

The custom program below writes one cacheline. It uses exact packing so the command position remains visible.


In [ ]:
def build_write_cacheline(
    bank: int,
    row: int,
    column: int,
    pattern: int,
) -> FinalProgram:
    p = ProgramBuilder(target=TARGET)
    p.LI(bank, "BAR")
    p.LI(row, "RAR")
    p.LI(column, "CAR")
    p.LI(TARGET.column_stride, "CASR")

    p.LI(pattern, "PATTERN_REG")
    for lane in range(TARGET.words_per_cacheline):
        p.LDWD("PATTERN_REG", lane)

    p.DRAM(PRE("BAR"), NOP(), NOP(), NOP())
    p.SLEEP(2)
    p.DRAM(ACT("BAR", "RAR"), NOP(), NOP(), NOP())
    p.SLEEP(2)
    p.DRAM(WR("BAR", "CAR"), NOP(), NOP(), NOP())
    p.SLEEP(9)
    p.DRAM(PRE("BAR"), NOP(), NOP(), NOP())
    p.SLEEP(3)
    return p.conclude()


custom_write = build_write_cacheline(
    bank=BANK,
    row=5,
    column=0,
    pattern=0xCAFEBABE,
)
print(custom_write.dry_run(max_instructions=10_000))
print(custom_write.trace_dram_commands().summarize_timings())


In [ ]:
CHECK_ROW = 5
CHECK_PATTERN = 0xCAFEBABE

readback = np.empty(ROW_WORDS, dtype=np.uint32)

with open_board(
    TARGET,
    pci_bdf=PCI_BDF,
    xdma_channel=XDMA_CHANNEL,
    host_interface=HostInterface.XDMA,
) as board:
    board.full_reset()
    board.execute([
        programs.write_row(
            BANK,
            CHECK_ROW,
            (0,) * TARGET.words_per_cacheline,
        ),
        build_write_cacheline(
            BANK,
            CHECK_ROW,
            0,
            CHECK_PATTERN,
        ),
        programs.read_row(BANK, CHECK_ROW),
    ])
    board.receive_into(readback, timeout=None)
    board.synchronize()

first_cacheline = readback[:TARGET.words_per_cacheline]
remaining_words = readback[TARGET.words_per_cacheline:]

assert np.all(first_cacheline == CHECK_PATTERN)
assert np.all(remaining_words == 0)
print("PASS: custom cacheline write")


## 7. JIT-Compiled Program Templates

The <code>@program_template</code> decorator traces a builder function, emits a native C++ plugin, compiles it, and caches the result. Later calls that vary only integer scalar arguments reuse the loaded specialization and patch those values. Non-integer arguments, including tuples and lists, are specialization inputs; changing them creates another specialization.

If the native JIT environment is unavailable, supported templates can fall back to interpreted construction. Unsupported template operations raise a compilation error instead of silently changing behavior. Use the run statistics to see what happened on the local system.


In [ ]:
@program_template
def build_write_cacheline_jit(
    bank: int,
    row: int,
    column: int,
    pattern: int,
) -> FinalProgram:
    p = ProgramBuilder(target=TARGET)
    p.LI(bank, "BAR")
    p.LI(row, "RAR")
    p.LI(column, "CAR")
    p.LI(TARGET.column_stride, "CASR")

    p.LI(pattern, "PATTERN_REG")
    for lane in range(TARGET.words_per_cacheline):
        p.LDWD("PATTERN_REG", lane)

    p.DRAM(PRE("BAR"), NOP(), NOP(), NOP())
    p.SLEEP(2)
    p.DRAM(ACT("BAR", "RAR"), NOP(), NOP(), NOP())
    p.SLEEP(2)
    p.DRAM(WR("BAR", "CAR"), NOP(), NOP(), NOP())
    p.SLEEP(9)
    p.DRAM(PRE("BAR"), NOP(), NOP(), NOP())
    p.SLEEP(3)
    return p.conclude()


In [ ]:
from drambender.api.jit import (
    get_jit_cache_dir,
    get_last_template_run_stats,
)

first_jit_program = build_write_cacheline_jit(
    BANK, 5, 0, 0x11111111
)
first_stats = get_last_template_run_stats()
assert first_stats is not None

second_jit_program = build_write_cacheline_jit(
    BANK, 6, 0, 0x22222222
)
second_stats = get_last_template_run_stats()
assert second_stats is not None

print("JIT cache:", get_jit_cache_dir())
print(
    "First call:",
    first_stats.mode,
    "cache_hit=",
    first_stats.cache_hit,
    "total_s=",
    f"{first_stats.total_s:.6f}",
)
print(
    "Second call:",
    second_stats.mode,
    "cache_hit=",
    second_stats.cache_hit,
    "total_s=",
    f"{second_stats.total_s:.6f}",
)


JIT diagnostics and cache controls live in <code>drambender.api.jit</code>:

- <code>get_last_template_run_stats()</code> reports the most recent construction path and timing.
- <code>get_jit_cache_dir()</code> reports the active on-disk cache.
- <code>set_jit_cache_dir(path)</code> selects another cache directory.
- <code>clear_template_caches()</code> clears in-memory specializations.
- <code>clear_template_caches(clear_disk=True)</code> also removes the active on-disk cache.


## 8. Advanced Composition: Custom Hammer Loop And Built-In Setup

One execute call can run an ordered sequence of built-in programs and custom JIT-generated programs. The custom template below hammers one aggressor row and then reads one victim row. Built-in programs initialize the two rows before it runs.

The preview uses a small hammer count so that dry-run and trace output remain bounded. The FPGA execution cell uses the full hammer count.


In [ ]:
@program_template
def build_hammer_and_read(
    bank: int,
    victim_row: int,
    aggressor_row: int,
    hammer_count: int,
) -> FinalProgram:
    p = ProgramBuilder(target=TARGET)
    p.alloc_reg("NUM_HMR")
    p.alloc_reg("HMR_COUNTER")

    p.LI(bank, "BAR")
    p.LI(TARGET.column_stride, "CASR")
    p.LI(aggressor_row, "RAR")
    p.LI(0, "HMR_COUNTER")
    p.LI(hammer_count, "NUM_HMR")

    p.LABEL("HAMMER")
    p.DRAM(PRE("BAR"), NOP(), NOP(), NOP())
    p.ADDI("HMR_COUNTER", 1, "HMR_COUNTER")
    p.DRAM(NOP(), NOP(), NOP(), ACT("BAR", "RAR"))
    p.BL("HMR_COUNTER", "NUM_HMR", "HAMMER")

    p.LI(victim_row, "RAR")
    p.DRAM(PRE("BAR"), NOP(), NOP(), NOP())
    p.LI(0, "CAR")
    p.SLEEP(2)
    p.DRAM(ACT("BAR", "RAR"), NOP(), NOP(), NOP())
    p.SLEEP(2)
    for _ in range(TARGET.cachelines_per_row):
        p.DRAM(RD("BAR", "CAR", icar=1), NOP(), NOP(), NOP())
        p.SLEEP(1)
    p.SLEEP(4)
    p.DRAM(PRE("BAR"), NOP(), NOP(), NOP())
    p.SLEEP(3)
    return p.conclude()


In [ ]:
preview_program = build_hammer_and_read(
    bank=BANK,
    victim_row=START_ROW,
    aggressor_row=START_ROW + 1,
    hammer_count=8,
)

preview_result = preview_program.dry_run(max_instructions=100_000)
preview_trace = preview_program.trace_dram_commands(
    max_instructions=100_000
)
assert not preview_trace.truncated

hammer_act_times = [
    event.time_ns
    for event in preview_trace.events
    if event.command == "ACT"
][:8]

print(preview_result)
print(
    "Consecutive hammer ACT intervals (ns):",
    np.diff(hammer_act_times),
)


In [ ]:
ADVANCED_HAMMER_COUNT = 500_000

victim = Row(
    physical_id=START_ROW,
    row_mapping="Linear",
    data_pattern=0x00000000,
)
aggressor = Row(
    physical_id=START_ROW + 1,
    row_mapping="Linear",
    data_pattern=0xFFFFFFFF,
)

program = build_hammer_and_read(
    bank=BANK,
    victim_row=victim.logical_id,
    aggressor_row=aggressor.logical_id,
    hammer_count=ADVANCED_HAMMER_COUNT,
)

with open_board(
    TARGET,
    pci_bdf=PCI_BDF,
    xdma_channel=XDMA_CHANNEL,
    host_interface=HostInterface.XDMA,
) as board:
    board.full_reset()
    board.execute([
        programs.write_row(
            BANK, victim.logical_id, victim.write_pattern
        ),
        programs.write_row(
            BANK, aggressor.logical_id, aggressor.write_pattern
        ),
        program,
    ])
    readback = np.empty(ROW_WORDS, dtype=np.uint32)
    board.receive_into(readback, timeout=None)
    board.synchronize()

row_pattern = np.asarray(victim.write_pattern, dtype=np.uint32)
expected = np.tile(row_pattern, ROW_WORDS // row_pattern.size)
mask = readback ^ expected
bitflips = int(
    np.unpackbits(mask.view(np.uint8), bitorder="little").sum()
)
print(f"Observed {bitflips} bit flips")


## 9. Retention Experiments

Both common retention workflows use the same API.

For a delay inside one DRAM Bender program, add one or more SLEEP instructions. Readback framing is independent of idle time, and <code>receive_into(timeout=None)</code> can wait across a long silent interval.

For a host-controlled delay, use two programs:

    board.execute(write_program)
    board.synchronize()
    time.sleep(retention_seconds)
    board.execute(read_program)
    board.receive_into(readback, timeout=None)
    board.synchronize()

Wait for the write program before starting the host delay. Configure automatic refresh with <code>board.set_aref(True)</code> or <code>board.set_aref(False)</code> according to the experiment.


## 10. Next Steps

- Built-in programs: configure a target and use <code>read_row</code>, <code>write_row</code>, <code>single_sided_rowhammer</code>, or <code>double_sided_rowhammer</code>. See [python/drambender/builtin_programs/](../python/drambender/builtin_programs/).
- Row mappings: inspect <code>drambender.rows.available_mappings()</code> and the exported <code>linear</code>, <code>sa0</code>, and <code>mi1</code> mappings.
- Pattern mappings: see [python/drambender/patterns/](../python/drambender/patterns/).
- Low-level program code: start at [ProgramBuilder](../python/drambender/api/program/builder.py) and the [instruction factories](../python/drambender/api/program/instructions.py).
- C++ API: see [examples/read_write.cpp](read_write.cpp) and [examples/single_sided_rowhammer.cpp](single_sided_rowhammer.cpp).
- General setup and hardware design: return to the [repository README](../README.md).

Every board session uses a context manager, so endpoint ownership is released when the cell exits. Begin a new session with <code>full_reset()</code> after an unclean process exit or abandoned in-flight program.
